# nb_21b_silver_history_retention

Silver, part 4. Two jobs that both need the full list of client versions.

**Job 1: retention (runs first).** Each region has its own time limit.
- A *prospect* is a client record with no purchase, repair or appointment. It expires after a number of months without any activity. The limit is set per region, with an optional override per country.
- A client who has bought something is never deleted automatically. After a long quiet period the record is only held for review because sales and finance records have their own legal rules.
- Expired prospects are removed from `bronze_crm_client` and `silver_crm_client`. Their raw copies in `ctl_quarantine` are blanked out.
- A tombstone (a one-way hash of the client id, no name or email) is stored in `ctl_erasure_tombstone`. If an old file is ever loaded again, `nb_20` sees the tombstone and refuses the row, so an erased client cannot come back.
- Every decision is written to `ctl_retention_log` as proof for an audit. It holds hashes only.

**Job 2: history.** `silver_crm_client` has one row per version of a client. This notebook turns those versions into `silver_client_history`, where every row says *from when to when* a set of values was true. It answers "what consent did we have for this client on 1 March?"

Tracked columns: consent to share across regions, marketing consent, email, phone, client segment, home boutique.

Order in the pipeline: `nb_20` -> `nb_21` -> `nb_21b` -> `nb_22`. Identity matching (`nb_22`) runs last, so it only ever sees clients that are still allowed to exist.

Input: `silver_crm_client`, `silver_pos_transaction`, `silver_ecom_order`, `silver_after_sales_ticket`, `silver_appointment`

Output: `silver_client_history`, `ctl_retention_rules`, `ctl_retention_log`, `ctl_erasure_tombstone`, plus rows in `ctl_dq_results` and `ctl_run_log`

(The retention limits below are illustrative settings for a fictional house. They are not legal advice and not a claim about any real company.)


In [ ]:
# ---- Settings -------------------------------------------------------------------------
# The date retention is measured from. None = the latest date found in the data (keeps re-runs repeatable).
# Set it to a fixed date such as "2026-09-30" to replay a past run.
AS_OF_DATE = None

# Prospects: months without any activity before the record expires. One value per region.
PROSPECT_MONTHS = {"americas": 36, "europe": 24, "asia_pacific": 36}
# Optional override for one country (two-letter code). It wins over the region value.
COUNTRY_PROSPECT_MONTHS = {"KR": 12}

# Clients who bought something: years without activity before the record is held for review. Never deleted here.
RELATIONSHIP_YEARS = 10

# The columns whose changes are kept as history.
TRACKED_COLUMNS = ["consent_share_cross_region", "consent_marketing", "email_hash", "phone_e164",
                   "client_segment", "home_boutique_id"]

In [ ]:
# ---- Setup ----------------------------------------------------------------------------
import uuid
from datetime import datetime, timezone, date
from functools import reduce

import pyspark.sql.functions as F
from pyspark.sql import Row, Window
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, LongType,
                               DoubleType, TimestampType, DateType)

SILVER_BATCH_ID = "S" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:6]
print("silver batch id:", SILVER_BATCH_ID)


def now_utc():
    return datetime.now(timezone.utc).replace(tzinfo=None)


dq_rows, run_rows, gaps = [], [], []


def add_dq(table, rule, severity, checked, failed):
    dq_rows.append(Row(batch_id=SILVER_BATCH_ID, layer="silver", table_name=table, rule=rule, severity=severity,
                       rows_checked=int(checked), rows_failed=int(failed),
                       pct_failed=round(100.0 * failed / checked, 3) if checked else 0.0, checked_at=now_utc()))


def write_table(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    return spark.table(name).count()


def rewrite_table(name, new_df):
    """Replace a table with new_df, where new_df was built FROM that same table.
    The result is staged in a helper table first, so the source is never read and written at the same time."""
    stage = name + "__stage"
    new_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(stage)
    spark.table(stage).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    spark.sql(f"DROP TABLE IF EXISTS {stage}")


def log_run(obj, status, rows, started, message, failure_class=None):
    run_rows.append(Row(batch_id=SILVER_BATCH_ID, layer="silver", object_name=obj, status=status, files_loaded=0,
                        rows_loaded=int(rows), started_at=started, finished_at=now_utc(),
                        failure_class=failure_class, message=message))

In [ ]:
# ---- Retention rules, written down as a table so anyone can read them ------------------
started = now_utc()
rule_rows = ([Row(scope="region", code=r, rule="prospect_inactive", limit_months=m) for r, m in PROSPECT_MONTHS.items()] +
             [Row(scope="country", code=c, rule="prospect_inactive", limit_months=m) for c, m in COUNTRY_PROSPECT_MONTHS.items()] +
             [Row(scope="all", code="*", rule="relationship_ended", limit_months=12 * RELATIONSHIP_YEARS)])
write_table(spark.createDataFrame(rule_rows), "ctl_retention_rules")
spark.table("ctl_retention_rules").orderBy("rule", "scope", "code").show(truncate=False)

In [ ]:
# ---- Who is a prospect, and when were they last active? ------------------------------
versions = spark.table("silver_crm_client")

# Activity is counted per client id. Client ids start with the region, so this is automatically per region.
act = reduce(lambda a, b: a.unionByName(b), [
    spark.table("silver_pos_transaction").select("client_id", F.col("event_date").alias("d")),
    spark.table("silver_ecom_order").select("client_id", F.col("event_date").alias("d")),
    spark.table("silver_after_sales_ticket").select("client_id", F.to_date("created_at").alias("d")),
    spark.table("silver_appointment").select("client_id", F.to_date("appointment_ts").alias("d"))])
last_activity = act.groupBy("client_id").agg(F.max("d").alias("last_activity_date"))

latest_w = Window.partitionBy("client_id").orderBy(F.col("updated_at").desc(), F.col("file_date").desc(), F.col("source_file").desc())
touched = versions.groupBy("client_id").agg(F.max("updated_at").cast("date").alias("last_update_date"))
clients = (versions.withColumn("rk", F.row_number().over(latest_w)).filter("rk = 1")
           .select("client_id", "region", "home_country", F.col("created_at").cast("date").alias("created_date"))
           .join(touched, "client_id").join(last_activity, "client_id", "left")
           .withColumn("is_prospect", F.col("last_activity_date").isNull())
           .withColumn("reference_date", F.greatest("created_date", "last_update_date", "last_activity_date")))

if AS_OF_DATE:
    as_of = date.fromisoformat(AS_OF_DATE)
else:
    as_of = max(d for d in [versions.agg(F.max(F.col("updated_at").cast("date"))).collect()[0][0],
                            act.agg(F.max("d")).collect()[0][0]] if d is not None)
print("retention is measured as of:", as_of, "| clients:", clients.count(), "| prospects:", clients.filter("is_prospect").count())

In [ ]:
# ---- Decide ---------------------------------------------------------------------------
region_months = F.create_map(*[x for r, m in PROSPECT_MONTHS.items() for x in (F.lit(r), F.lit(m))])
country_months = (F.create_map(*[x for c, m in COUNTRY_PROSPECT_MONTHS.items() for x in (F.lit(c), F.lit(m))])
                  if COUNTRY_PROSPECT_MONTHS else F.create_map())
limit_months = F.when(F.col("is_prospect"),
                      F.coalesce(F.element_at(country_months, F.col("home_country")), F.element_at(region_months, F.col("region"))))\
                .otherwise(F.lit(12 * RELATIONSHIP_YEARS))
decided = (clients.withColumn("limit_months", limit_months.cast("int"))
           .withColumn("expires_on", F.add_months(F.col("reference_date"), F.col("limit_months")))
           .withColumn("is_expired", F.col("expires_on") < F.lit(as_of))
           .withColumn("rule", F.when(F.col("is_prospect"), "prospect_inactive").otherwise("relationship_ended"))
           .withColumn("action", F.when(~F.col("is_expired"), None)
                                  .when(F.col("is_prospect"), "purged").otherwise("held_for_review")))
expired = decided.filter("is_expired").cache()
purge_ids = expired.filter("action = 'purged'").select("client_id").cache()
n_purge = purge_ids.count()
add_dq("silver_crm_client", "prospect_expired", "info", clients.filter("is_prospect").count(), n_purge)
add_dq("silver_crm_client", "relationship_held_for_review", "info", clients.filter("not is_prospect").count(),
       expired.filter("action = 'held_for_review'").count())
expired.groupBy("region", "home_country", "rule", "action").count().orderBy("region", "home_country").show(40, truncate=False)
print("clients to purge:", n_purge)

In [ ]:
# ---- Purge, leave proof, leave a tombstone --------------------------------------------
started = now_utc()
n_before = spark.table("silver_crm_client").select("client_id").distinct().count()
purged_ids_list = [r["client_id"] for r in purge_ids.collect()]      # kept in memory only, for the checks below

if n_purge > 0:
    # 1) tombstones and the audit log. Hashes only: no name, email, phone or client id is written.
    tomb = (expired.filter("action = 'purged'")
            .select(F.sha2("client_id", 256).alias("client_id_hash"), "region", "rule",
                    F.current_timestamp().alias("erased_at"), F.lit(SILVER_BATCH_ID).alias("batch_id")))
    tomb.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("ctl_erasure_tombstone")

    # 2) remove the client from the CRM tables
    for t in ["bronze_crm_client", "silver_crm_client"]:
        rewrite_table(t, spark.table(t).join(purge_ids, "client_id", "left_anti"))

    # 3) blank the raw copy of the client in the quarantine table (rows that failed a rule earlier)
    q = spark.table("ctl_quarantine")
    hit = (F.col("source_table") == "bronze_crm_client") & F.col("record_key").isin(purged_ids_list)
    rewrite_table("ctl_quarantine", q.withColumn("raw_record", F.when(hit, F.lit('{"redacted":"erased_client"}')).otherwise(F.col("raw_record")))
                                     .withColumn("record_key", F.when(hit, F.sha2(F.col("record_key"), 256)).otherwise(F.col("record_key"))))

# the audit log gets one row per decision (purged or held), hashes only
LOG_COLS = ["batch_id", "client_id_hash", "region", "home_country", "rule", "action", "limit_months",
            "reference_date", "as_of_date", "decided_at"]
log_df = (expired.select(F.lit(SILVER_BATCH_ID).alias("batch_id"), F.sha2("client_id", 256).alias("client_id_hash"),
                         "region", "home_country", "rule", "action", "limit_months", "reference_date",
                         F.lit(as_of).cast("date").alias("as_of_date"), F.current_timestamp().alias("decided_at"))
          .select(*LOG_COLS))
log_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("ctl_retention_log")

n_after = spark.table("silver_crm_client").select("client_id").distinct().count()
if n_before - n_after != n_purge:
    gaps.append("clients removed from silver_crm_client")
n_held = expired.filter(F.col("action") == "held_for_review").count()
log_run("retention", "success" if n_before - n_after == n_purge else "failed", n_purge, started,
        f"purged={n_purge} held_for_review={n_held} as_of={as_of}")
print(f"clients before: {n_before} | purged: {n_purge} | clients after: {n_after}")

In [ ]:
# ---- History: one row per period during which the tracked values did not change -------
started = now_utc()
src = spark.table("silver_crm_client")
w = Window.partitionBy("client_id").orderBy("updated_at", "file_date", "source_file")
h = src.select("client_id", "region", "created_at", "updated_at", "file_date", "source_file", *TRACKED_COLUMNS)\
       .withColumn("_n", F.row_number().over(w))
changed = []
for c in TRACKED_COLUMNS:
    h = h.withColumn(f"_prev_{c}", F.lag(c).over(w))
    changed.append((c, ~F.col(c).eqNullSafe(F.col(f"_prev_{c}"))))
any_changed = reduce(lambda a, b: a | b, [cond for _, cond in changed])
h = h.withColumn("_changed_cols", F.concat_ws(",", *[F.when(cond, F.lit(c)) for c, cond in changed]))
# keep the first version and every version where a tracked value really changed.
# Resent copies and name-only edits change nothing that is tracked, so they drop out here.
h = h.filter((F.col("_n") == 1) | any_changed)
w2 = Window.partitionBy("client_id").orderBy("updated_at", "file_date", "source_file")
hist = (h.withColumn("version_number", F.row_number().over(w2))
          .withColumn("valid_from", F.col("updated_at"))
          .withColumn("valid_to", F.lead("updated_at").over(w2))
          .withColumn("is_current", F.col("valid_to").isNull())
          .withColumn("changed_columns", F.when(F.col("version_number") == 1, F.lit("first_seen")).otherwise(F.col("_changed_cols")))
          .select("client_id", "region", "version_number", "valid_from", "valid_to", "is_current", "changed_columns",
                  *TRACKED_COLUMNS, "created_at", F.lit(SILVER_BATCH_ID).alias("silver_batch_id")))
n_hist = write_table(hist, "silver_client_history")
log_run("silver_client_history", "success", n_hist, started, f"rows={n_hist}")
print("silver_client_history:", n_hist, "rows for", spark.table("silver_client_history").select("client_id").distinct().count(), "clients")

In [ ]:
# ---- Checks on the history, then reconciliation ---------------------------------------
hist = spark.table("silver_client_history")
n_clients_now = spark.table("silver_crm_client").select("client_id").distinct().count()
n_hist_clients = hist.select("client_id").distinct().count()
cur_per_client = hist.filter("is_current").groupBy("client_id").count()
not_one_current = cur_per_client.filter("count != 1").count() + (n_hist_clients - cur_per_client.count())
overlap = hist.filter(F.col("valid_to").isNotNull() & (F.col("valid_to") <= F.col("valid_from"))).count()
add_dq("silver_client_history", "client_without_exactly_one_current_row", "reject", n_hist_clients, not_one_current)
add_dq("silver_client_history", "period_ends_before_it_starts", "reject", hist.count(), overlap)
if n_clients_now != n_hist_clients:
    gaps.append("clients in silver_crm_client vs silver_client_history")
if not_one_current:
    gaps.append("clients without exactly one current history row")
if overlap:
    gaps.append("history periods that end before they start")
consent_changes = hist.filter(F.array_contains(F.split("changed_columns", ","), "consent_share_cross_region")).count()
add_dq("silver_client_history", "consent_share_changed", "info", hist.count(), consent_changes)

DQ_SCHEMA = StructType([
    StructField("batch_id", StringType()), StructField("layer", StringType()),
    StructField("table_name", StringType()), StructField("rule", StringType()),
    StructField("severity", StringType()), StructField("rows_checked", LongType()),
    StructField("rows_failed", LongType()), StructField("pct_failed", DoubleType()),
    StructField("checked_at", TimestampType())])
RUN_SCHEMA = StructType([
    StructField("batch_id", StringType()), StructField("layer", StringType()),
    StructField("object_name", StringType()), StructField("status", StringType()),
    StructField("files_loaded", IntegerType()), StructField("rows_loaded", LongType()),
    StructField("started_at", TimestampType()), StructField("finished_at", TimestampType()),
    StructField("failure_class", StringType()), StructField("message", StringType())])
spark.createDataFrame(dq_rows, schema=DQ_SCHEMA).write.format("delta").mode("append") \
     .option("mergeSchema", "true").saveAsTable("ctl_dq_results")
spark.createDataFrame(run_rows, schema=RUN_SCHEMA).write.format("delta").mode("append") \
     .option("mergeSchema", "true").saveAsTable("ctl_run_log")

if gaps:
    raise RuntimeError("Silver part 4 checks failed for: " + ", ".join(gaps))
print("silver part 4 finished, batch", SILVER_BATCH_ID)

## Check the result
Read-only cells. Run them after the load.

In [ ]:
# What retention decided, by region and country
(spark.table("ctl_retention_log").filter(F.col("batch_id") == SILVER_BATCH_ID)
      .groupBy("region", "home_country", "rule", "action", "limit_months").count().orderBy("region", "home_country").show(40, truncate=False))

In [ ]:
# Erasure proof: none of the purged clients may still exist in any table that holds client ids.
# nb_22 (identity) is expected to show leftovers the FIRST time, until it is run again after this notebook.
ids = purged_ids_list
rows = []
for t, cols in [("bronze_crm_client", ["client_id"]), ("silver_crm_client", ["client_id"]), ("silver_client_history", ["client_id"]),
                ("silver_client_identity", ["client_id"]), ("silver_identity_pairs", ["client_id_a", "client_id_b"]),
                ("ctl_quarantine", ["record_key"])]:
    if not spark.catalog.tableExists(t):
        rows.append((t, -1)); continue
    d = spark.table(t)
    cond = reduce(lambda a, b: a | b, [F.col(c).isin(ids) for c in cols]) if ids else F.lit(False)
    rows.append((t, d.filter(cond).count()))
spark.createDataFrame(rows, ["table", "rows_still_referring_to_purged_clients"]).show(truncate=False)
print("-1 = table does not exist yet. A number above 0 in silver_client_identity / silver_identity_pairs means: run nb_22 again.")

In [ ]:
# Tombstones: how many clients are blocked from coming back
spark.table("ctl_erasure_tombstone").groupBy("region", "rule").count().show()
spark.table("ctl_erasure_tombstone").select("client_id_hash", "region", "erased_at").show(3, truncate=30)

In [ ]:
# History: how many versions do clients have, and what changed?
hist = spark.table("silver_client_history")
hist.groupBy("version_number").count().orderBy("version_number").show()
(hist.filter("version_number > 1").groupBy("changed_columns").count().orderBy(F.col("count").desc()).show(truncate=False))

In [ ]:
# "What did we know on a given date?" Pick a client whose consent changed and ask about three dates.
c = (spark.table("silver_client_history").filter(F.array_contains(F.split("changed_columns", ","), "consent_share_cross_region"))
     .orderBy("client_id").select("client_id", "valid_from").first())
cid = c["client_id"]
print("client:", cid)
spark.table("silver_client_history").filter(F.col("client_id") == cid).select(
    "version_number", "valid_from", "valid_to", "is_current", "changed_columns", "consent_share_cross_region").orderBy("version_number").show(truncate=False)
for label, d in [("before the change", str(c["valid_from"] - __import__("datetime").timedelta(days=2))[:10]), ("after the change", str(c["valid_from"] + __import__("datetime").timedelta(days=2))[:10])]:
    r = spark.sql(f"""SELECT consent_share_cross_region FROM silver_client_history
                      WHERE client_id = '{cid}' AND valid_from <= TIMESTAMP '{d} 23:59:59' AND (valid_to IS NULL OR valid_to > TIMESTAMP '{d} 23:59:59')""").collect()
    print(f"consent to share on {d} ({label}):", r[0][0] if r else "client did not exist yet")

In [ ]:
# Reconciliation
spark.createDataFrame([
    ("clients in silver_crm_client", n_clients_now), ("clients in silver_client_history", n_hist_clients),
    ("clients without exactly one current row", not_one_current), ("history periods that end before they start", overlap),
    ("clients purged this run", n_purge)], ["check", "value"]).show(truncate=False)

In [ ]:
spark.table("ctl_run_log").filter("layer = 'silver' and object_name in ('retention','silver_client_history')").orderBy(F.col("finished_at").desc()).select("object_name","status","rows_loaded","message").show(4, truncate=False)